# 1. Sen4AgriNet: overnight candidate batches and annual TIFFs

Retain the existing source-label, geometry, identity and observation-year checks. Target up to 250 eligible parcels per available crop in the current round before preparing masks and requesting imagery. Upstream reserved splits remain excluded from development sampling. Reference preparation defaults to saved checkpoints only, with a 350-patch cap and targets of at most 250 carrot and 250 potato parcels. Cabbage comes from the completed FranceCrops collection. It submits available candidates even if carrot is short. It checks complete scanned patches, excludes mixed/conflicting parcel groups and does not claim full-release coverage. No new reference patch transfers occur with `reference_cached_only = True`. To expand later, set it to False and choose an explicit patch cap; reaching the cap publishes the available candidates and reports deficits.

Each selected tile retains all 24 half-month periods, so four-month windows can change later. After the candidate batches complete, notebook 04 applies the existing timing/quality checks and reports usable pixels and parcels by locked train/validation/test partition. Subsequent batches fill only measured deficits.

Annual TIFFs are organized by source, year, campaign and round. Each contains 168 labelled bands; sampling masks are separate. Opening the notebook starts no work. Restart Kernel → Run All prepares/rebuilds masks and submits or collects the current overnight batches. Batch output reports saved data and cloud job states; completion percentages do not represent elapsed processing time.


## 1. Fresh-clone setup

Choose a Python 3.12+ notebook kernel (3.13 recommended), then **Run All**. The first cell makes the cloned package importable, installs missing dependencies in that kernel, and creates `settings.local.json` with portable local paths if absent. Existing settings are preserved.

If no Earth Engine project is configured, enter your registered project ID once when prompted. It is saved locally. Complete the later Google sign-in prompt. No saved NetCDFs, inventories or imagery are required; subsequent stages obtain their inputs automatically.

**Output:** kernel setup, local folders and project configuration. Full-study reference and satellite collection switches are enabled for Run All.


In [1]:
# Standard-library setup runs before scientific imports, so a fresh clone needs no saved data.
from pathlib import Path
import sys

notebook_repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebook_source_folder = str(notebook_repository_root / "src")
if notebook_source_folder not in sys.path:
    sys.path.insert(0, notebook_source_folder)
from crop_pipeline import notebook_setup
notebook_setup.prepare_environment(notebook_repository_root)

from pathlib import Path
import json
import os
from crop_pipeline.sources import annual
from crop_pipeline.sources import sen4agrinet as sen4agrinet_references

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
dataset_folder = Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./ground_truth"))).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
reference_root = Path(local_settings.get("sen4agrinet_reference_root", str(annual_archive_root / "sen4agrinet_references"))).expanduser().resolve()
from crop_pipeline.sources import adaptive
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = 250  # Maximum per Sen4AgriNet crop, subject to cached availability
reference_target_crops = ("carrot", "potato")  # Cabbage comes from the completed FranceCrops batch
reference_scan_max_patches = 350  # Hard cap on patches included in this reference subset
reference_cached_only = True  # Submit from saved checkpoints now; no new reference patch transfers
earth_engine_project = os.environ.get("EE_PROJECT", local_settings.get("earth_engine_project", ""))

# Run All scans references only until each crop has enough eligible physical parcels.
retrieve_reference_masks = True
rebuild_field_inventory = True
run_satellite_collection = True
# Small request groups reduce overhead without changing the imagery recipe.
collection_backend = "batch"  # Earth Engine assets; rerun to collect finished jobs
archive_format = "pixel_tables"  # Annual selected-pixel Parquet; raster archives remain available
tile_start = 0
tile_count = None  # Adaptive candidates are already bounded by parcel budgets
download_workers = 2
intervals_per_request = 6
authenticate_now = True

# None reuses the first pinned release. Set a specific release SHA only to start another release.
dataset_revision = None
print("Existing raw files, read-only reuse:", dataset_folder)
print("Versioned full-reference outputs:", reference_root)
print("Local satellite archive:", annual_archive_root)

# Resolve account setup before large reference transfers start.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)

from crop_pipeline import progress as pipeline_progress
notebook_progress = pipeline_progress.start_notebook(
    "01_sen4agrinet_preparation",
    ['Certificates', 'Discover reference patches', 'Extract reference masks', 'Prepare eligible parcels', 'Collect annual imagery'],
    report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "satellite_collection": globals().get("run_satellite_collection"), "training_mode": globals().get("training_mode")},
)


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python


/Users/josedavid/Documents/programming/external-crop-data-classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Existing raw files, read-only reuse: /Users/josedavid/Documents/programming/external-crop-data-classification/ground_truth
Versioned full-reference outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/sen4agrinet_references
Local satellite archive: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive


01_sen4agrinet_preparation: stages:   0%|          | 0/5 [00:01<?, ?stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]

### HTTPS certificates

The bootstrap configures trust before the first request. This cell displays the trusted-certificate count before reference discovery; HTTPS verification remains enabled.


In [2]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Certificates')

import ssl
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])

pipeline_progress.end_stage("completed")


01_sen4agrinet_preparation: stages:  20%|██        | 1/5 [00:01<00:05,  1.49s/stage, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]

Trusted certificates: 121


## 2. Discover every year and tile

List every NetCDF patch under the pinned repository's `data/` tree. There is no year, tile or candidate-budget filter. The same pinned release is reused across restarts.

**Output:** release SHA, patch count, years and year/tile combinations. The displayed full NetCDF size is context only: it is **not** the planned transfer. Reference-only transfer size depends on mask compression and HDF5 layout and cannot be known exactly before reading. The next stage reports actual received bytes.


In [3]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Discover reference patches')

repository_manifest = sen4agrinet_references.discover_release(
    reference_root, enabled=retrieve_reference_masks, revision=dataset_revision,
)
reference_run_folder = sen4agrinet_references.run_folder(reference_root, repository_manifest)
print("Pinned dataset release:", repository_manifest["revision"])
print("NetCDF patches:", len(repository_manifest["files"]))
print("Years:", repository_manifest["years"])
print("Year/tile combinations:", len(repository_manifest["year_tile_pairs"]))
print("Full NetCDF size, NOT the planned transfer (GiB):", round(repository_manifest["total_source_bytes"] / 1024**3, 2))
print("Largest upstream NetCDF, NOT downloaded whole (GiB):", round(max(record["expected_bytes"] for record in repository_manifest["files"]) / 1024**3, 2))
print("Reference run:", reference_run_folder)

pipeline_progress.end_stage("completed")


01_sen4agrinet_preparation: stages:  40%|████      | 2/5 [00:01<00:02,  1.32stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]                    

Pinned dataset release: 4eb44157012b9ec251767721d6bae55cf0c564e2
NetCDF patches: 10013
Years: [2019, 2020]
Year/tile combinations: 16
Full NetCDF size, NOT the planned transfer (GiB): 261.75
Largest upstream NetCDF, NOT downloaded whole (GiB): 0.06
Reference run: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/sen4agrinet_references/releases/4eb44157012b9ec251767721d6bae55cf0c564e2/inventories/4710076785993f11


## 3. Read reference masks without downloading full NetCDF files

Reuse verified local files when available. Otherwise, request small byte ranges for HDF5 metadata, the label mask and its coordinates. Read the parcel mask only when target labels are present. Spectral arrays are not accessed. A small amount of adjacent data may share requested byte blocks.

Save compressed mask-only NetCDF caches and extracted-reference checkpoints locally. Valid patch checkpoints avoid further reads; mask caches can survive feature-extractor changes. No complete imagery NetCDF is staged or deleted in this mode. If a server ignores HTTP Range, stop before reading its body; there is **no automatic full-file fallback**.

**Eligibility stays unchanged:** crop codes, valid masks, 10 m grid/coordinate checks, parcel IDs, geometry validation and ambiguous-group exclusions. Each visited patch must pass extraction checks; a failure stops the run for safe retry. Scanning stops at the patch cap or target count; cache-only mode uses saved checkpoints and then continues with the available candidates. Previously verified patch checkpoints are reused. Unseen patches have not been audited for label conflicts; the saved inventory explicitly records subset coverage.

**Output:** progress per patch, eligible crop counts, excluded groups, actual received reference MiB and checkpoint reuse count. The source is pinned by release/path and checked HTTP offsets/size. Partial reads cannot verify the entire upstream NetCDF checksum; downloaded mask caches have their own SHA-256 integrity checks.


In [4]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Extract reference masks')

from crop_pipeline.sources import bounded_references
reference_summary = bounded_references.prepare(
    repository_manifest, reference_root, dataset_folder,
    parcels_per_crop=candidate_parcels_per_crop,
    seed=adaptive.DEFAULT_POLICY["seed"],
    download_enabled=retrieve_reference_masks,
    scan_enabled=rebuild_field_inventory,
    target_crops=reference_target_crops,
    max_patches=reference_scan_max_patches,
    cached_only=reference_cached_only,
)
print("Reference scope:", reference_summary.get("scope", "verified_full_release"))
print("Reference stop reason:", reference_summary.get("stop_reason", "full_inventory_reused"))
print("Reference deficits:", reference_summary.get("deficits", {}))
print("Cabbage source: FranceCrops (not requested from Sen4AgriNet)")
print("New reference patch transfers enabled:", not reference_cached_only)
print("Reference scan:", reference_summary["status"])
print("Files scanned:", reference_summary["files_scanned"])
print("Eligible crop counts:", reference_summary["crop_counts"])
print("Excluded ambiguous parcel groups:", reference_summary["excluded_parcel_groups"])
print("Scanned reference fields:", reference_summary["field_root"])

transfer_path = Path(reference_summary["field_root"]).parent / "last_scan_transfer.json"
if transfer_path.exists():
    transfer = json.loads(transfer_path.read_text())
    print("Reference bytes received this scan (MiB):", round(transfer["network_bytes"] / 1024**2, 2))
    print("Patches reused from checkpoints:", transfer["patches_reused_from_checkpoints"])

pipeline_progress.end_stage("completed")


01_sen4agrinet_preparation: stages:  60%|██████    | 3/5 [00:01<00:01,  1.95stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]                 

Reusing bounded reference inventory: 350 patches.
Reference scope: bounded_training_reference_subset
Reference stop reason: patch_cap_reached
Reference deficits: {'carrot': 215, 'potato': 0}
Cabbage source: FranceCrops (not requested from Sen4AgriNet)
New reference patch transfers enabled: False
Reference scan: complete
Files scanned: 350
Eligible crop counts: {'carrot': {'fragments': 36, 'parcel_groups': 35}, 'potato': {'fragments': 702, 'parcel_groups': 698}}
Excluded ambiguous parcel groups: 0
Scanned reference fields: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/sen4agrinet_references/releases/4eb44157012b9ec251767721d6bae55cf0c564e2/inventories/4710076785993f11/bounded/710faa22469fd827/fields
Reference bytes received this scan (MiB): 0.0
Patches reused from checkpoints: 350


## Prepare a bounded candidate parcel batch

Choose a reproducible crop-balanced subset, spread across geographic/year strata, before rasterization. Keep the existing inward buffer and screen contested locations against all reference parcels, including unselected neighbors. Only selected parcels receive masks and satellite requests. Printed estimates describe this candidate batch, not the full reference catalogue.


In [5]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Prepare eligible parcels')

reference_features, inventory_provenance = annual.load_sen4agrinet_references(Path(reference_summary["field_root"]))
adaptive_batches = adaptive.prepare_round_batches(
    "sen4agrinet", reference_features, annual_archive_root, report_folder,
    provenance={"dataset": sen4agrinet_references.DATASET_NAME,
                "dataset_revision": repository_manifest["revision"],
                "coverage": reference_summary.get("scope", "verified_full_release"),
                "reference_stop_reason": reference_summary.get("stop_reason", "full_inventory_reused"),
                "reference_scanned_files": reference_summary.get("scanned_files", []),
                "repository_files_scanned": reference_summary["files_scanned"],
                "reference_summary": str(Path(reference_summary["field_root"]).parent / "completed_inventory.json"),
                "inventories": inventory_provenance},
    inward_buffer_metres=0, parcels_per_crop=candidate_parcels_per_crop,
    candidate_crops=reference_target_crops,
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Batch parcels to resume/submit:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles across active batches:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
print("Satellite estimate:", annual_plan["estimate"] if annual_plan else "No new candidates required")
print("Reference years:", list(annual_plan["years"]) if annual_plan else [])

pipeline_progress.set_workload({"folder": str(annual_archive_root), "source_revision": annual_plan["revision"] if annual_plan else "targets_met_or_exhausted", "satellite_collection": run_satellite_collection})

pipeline_progress.end_stage("completed")


01_sen4agrinet_preparation: stages:  80%|████████  | 4/5 [00:01<00:00,  2.35stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]                  

Adaptive round 1: 285 sen4agrinet candidate parcels; annual histories retained, usable targets checked after four-month selection.
Round 1: target up to 250 candidates per available crop. Resume/submit 1 batches; previous jobs and locked partitions are preserved.
Batch parcels to resume/submit: 285
Tiles across active batches: 338
Satellite estimate: {'parcels': 285, 'tiles': 338, 'composites': 8112, 'raw_raster_bytes': 86079168, 'raw_raster_gib': 0.08, 'recommended_free_bytes': 172158336, 'available_bytes': 58542260224, 'note': 'Uncompressed float32 rasters; allow additional disk for summaries, features and models. Compression varies.'}
Reference years: ['2019', '2020']


## Collect annual histories for the selected candidates

Earth Engine retains the existing cloud masks, daily mosaics, bilinear band alignment and half-month medians. A completed batch means its planned annual records and daily summaries are available; usable four-month counts are assessed afterward. Re-running a pending batch collects existing jobs without expanding its candidate selection.


### Collect a bounded candidate batch

This notebook targets up to 250 eligible carrot parcels and 250 potato parcels in the current round, using the capped reference subset, spread across geographic/year strata, before rasterization. Annual histories remain available for changing the four-month window. Re-run the same source notebook to collect its existing jobs. After the candidate batches finish, run notebook 04 to count complete usable windows and write deficits. The next run selects new parcels only for deficient crop/partition combinations. Targets are 1,200 training pixels from at least 60 parcels, 400 validation pixels from at least 20 parcels, and 400 reserved-test pixels from at least 20 parcels per crop, with at most 20 retained pixels per parcel. These are initial sampling targets, not a guarantee of model accuracy.

Completed output is organized under `sen4agrinet_work/annual_tifs/<source>/<year>/<campaign>/round_<number>/`. One TIFF per tile contains 168 labelled bands (24 periods × seven measurements). Surrounding unselected pixels are nodata. The compact Parquet files remain the export transport/cache; window analysis reads the organized annual TIFFs. Sampling masks are separate and regenerated during source preparation if removed.

No 5,000-tile or full-catalogue download is the default. Source batches and earlier completed rounds remain separate and are combined by notebook 04. Upstream reserved splits are excluded from development sampling, and locked parcel assignments do not change when new batches are added.

**Overnight setting:** `candidate_parcels_per_crop = 250` (up to 500 parcels across carrot and potato, subject to cached availability and measured deficits; FranceCrops supplies cabbage). Existing selections count toward the round target; a separate extension preserves earlier jobs and locked partitions. This is a workload budget, not a guaranteed eight hours of Earth Engine processing or EECU time. Reference preparation and submission require the laptop to stay on. Wait for Run All to return normally and inspect job counts; submitted jobs continue while the laptop is off. Rerun later to collect and create annual TIFFs.


In [6]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Collect annual imagery')

collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    tile_start=tile_start, tile_count=tile_count,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
if collection_result.get("progress_by_year"):
    print({key: collection_result[key] for key in ("status", "selected_batches", "registry") if key in collection_result})
else:
    print(collection_result)

pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")
pipeline_progress.finish_notebook()


01_sen4agrinet_preparation: Collect annual imagery:  80%|████████  | 4/5 [00:04<00:01,  1.12s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]

sen4agrinet 2019: selecting tiles 1–258 of 258


01_sen4agrinet_preparation: Collect annual imagery:  80%|████████  | 4/5 [00:39<00:09,  9.92s/stage, elapsed 0m 39s; notebook ETA learning (future stages not yet timed)]

Scene inventories: 25/258 (35.5s)


01_sen4agrinet_preparation: Collect annual imagery:  80%|████████  | 4/5 [01:02<00:15, 15.71s/stage, elapsed 1m 02s; notebook ETA learning (future stages not yet timed)]

Scene inventories: 50/258 (58.8s)


... (28 outputs left out)

01_sen4agrinet_preparation: Collect annual imagery:  80%|████████  | 4/5 [07:07<01:46, 106.80s/stage, elapsed 7m 07s; notebook ETA learning (future stages not yet timed)]

Complete tiles: 0/80 (0.0%)
  Pixel data: 0/80 | Daily summaries: 0/80
  Half-month periods verified locally: 0/1920 | Pending: 1920
  Jobs: 0 running, 3 queued, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
  Completion measures saved results, not a percentage of processing time. ETA: unavailable.
sen4agrinet 2020: selected range pending; rerun the same range later. Ledger: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/adaptive_training/829e0cec79b8/round_0001/overnight_250/sen4agrinet/runs/full_year_all_parcels/a3f07a295c7d051a/batch/pixel_tables/2020.json


01_sen4agrinet_preparation: stages:  80%|████████  | 4/5 [07:07<01:46, 106.82s/stage, elapsed 7m 07s; notebook ETA learning (future stages not yet timed)]                

{'status': 'pending_retry'}
01_sen4agrinet_preparation: 7m 07s elapsed; timing report: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/progress/01_sen4agrinet_preparation/runs/20261107T064301_811c1f88.json


## 6. Continue after all source collections finish

Finish [FranceCrops](02_francecrops_preparation.ipynb) and [CropHarvest](03_cropharvest_preparation.ipynb), then run [04 Four-month windows](04_four_month_windows.ipynb), [05 Feature engineering](05_feature_engineering.ipynb), and [06 Model training](06_model_training.ipynb).

Notebook 04 requires all three completed handoffs. It applies timing and quality checks to the saved annual archive and chooses the earliest complete usable window per parcel. Old results remain in their original folders. January–December archives still require an explicit adjacent-year extension for boundary-crossing windows.
